# 1.4 CNN: gėlių nuotraukų klasifikavimas

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/gabrielbaranovskij/krepselisai/blob/main/notebooks/1_4_CNN_geliu_klasifikavimas.ipynb)

**Komanda:** Edvin Vilkanec, Gabriel Baranovskij · VILNIUSTECH

Dėstytojo *Colab CNN* pavyzdį perdarome su kitais duomenimis: **3670 tikrų spalvotų gėlių nuotraukų**, 5 klasės (ramunė, kiaulpienė, rožė, saulėgrąža, tulpė). Konvoliucinį tinklą (CNN) mokome **nuo nulio**.

> Colab: *Runtime → Change runtime type → GPU*, tada *Run all*.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import layers

print("TensorFlow", tf.__version__, "| GPU:", tf.config.list_physical_devices("GPU"))

## 1. Duomenys

Nuotraukos atsisiunčiamos ir suskirstomos į aplankus pagal klasę.

In [ ]:
import pathlib, tarfile, urllib.request

URL = "https://storage.googleapis.com/download.tensorflow.org/example_images/flower_photos.tgz"
if not pathlib.Path("flower_photos").exists():
    urllib.request.urlretrieve(URL, "flower_photos.tgz")
    tarfile.open("flower_photos.tgz").extractall(".")
data_dir = pathlib.Path("flower_photos")

for d in sorted(p for p in data_dir.iterdir() if p.is_dir()):
    print(f"{d.name:12s} {len(list(d.glob('*.jpg')))} nuotr.")

In [ ]:
IMG, BATCH = 180, 32
train_ds, val_ds = tf.keras.utils.image_dataset_from_directory(
    data_dir, validation_split=0.2, subset="both", seed=123, image_size=(IMG, IMG), batch_size=BATCH)
klases = train_ds.class_names
print(klases)

plt.figure(figsize=(10, 5))
for x, y in train_ds.take(1):
    for i in range(10):
        plt.subplot(2, 5, i + 1); plt.imshow(x[i].numpy().astype("uint8")); plt.title(klases[y[i]]); plt.axis("off")
plt.tight_layout(); plt.show()

train_ds = train_ds.cache().shuffle(1000).prefetch(tf.data.AUTOTUNE)
val_ds = val_ds.cache().prefetch(tf.data.AUTOTUNE)

## 2. CNN modelis

| Sluoksnis | Ką daro |
|---|---|
| `RandomFlip`, `RandomRotation`, `RandomZoom` | **augmentacija**: kiekvieną epochą nuotraukos šiek tiek pasukamos ir apverčiamos, kad modelis mažiau persimokytų |
| `Rescaling(1/255)` | pikseliai 0–255 → 0–1 |
| `Conv2D` | filtrai 3×3 slenka per nuotrauką ir randa požymius: pirmi sluoksniai kraštus, gilesni formas ir žiedlapius |
| `MaxPooling2D` | sumažina vaizdą perpus, palieka stipriausius požymius |
| `Dropout` | mokymo metu atsitiktinai išjungia dalį neuronų prieš persimokymą |
| `Flatten` → `Dense` | iš požymių priima sprendimą; paskutinis sluoksnis turi 5 išvestis, po vieną klasei |

In [ ]:
modelis = tf.keras.Sequential([
    tf.keras.Input(shape=(IMG, IMG, 3)),
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
    layers.Rescaling(1 / 255),
    layers.Conv2D(16, 3, padding="same", activation="relu"), layers.MaxPooling2D(),
    layers.Conv2D(32, 3, padding="same", activation="relu"), layers.MaxPooling2D(),
    layers.Conv2D(64, 3, padding="same", activation="relu"), layers.MaxPooling2D(),
    layers.Dropout(0.2),
    layers.Flatten(),
    layers.Dense(128, activation="relu"),
    layers.Dense(len(klases)),
])
modelis.compile(optimizer="adam",
                loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
                metrics=["accuracy"])
modelis.summary()

## 3. Mokymas

In [ ]:
EPOCHOS = 15
istorija = modelis.fit(train_ds, validation_data=val_ds, epochs=EPOCHOS)

h = istorija.history
fig, ax = plt.subplots(1, 2, figsize=(12, 3.5))
ax[0].plot(h["accuracy"], label="mokymas"); ax[0].plot(h["val_accuracy"], label="validacija"); ax[0].set_title("Tikslumas")
ax[1].plot(h["loss"], label="mokymas"); ax[1].plot(h["val_loss"], label="validacija"); ax[1].set_title("Nuostoliai")
for a in ax: a.set_xlabel("epocha"); a.legend()
plt.tight_layout(); plt.show()
print(f"Galutinis validacijos tikslumas: {h['val_accuracy'][-1]:.3f}")

## 4. Prognozės ir klaidų matrica

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay

y_true, y_pred = [], []
for x, y in val_ds:
    y_true += list(y.numpy()); y_pred += list(np.argmax(modelis.predict(x, verbose=0), axis=1))
ConfusionMatrixDisplay.from_predictions(y_true, y_pred, display_labels=klases, xticks_rotation=45, cmap="Greens")
plt.title("Klaidų matrica (validacija)"); plt.tight_layout(); plt.show()

x, y = next(iter(val_ds))
p = tf.nn.softmax(modelis.predict(x, verbose=0)).numpy()
plt.figure(figsize=(12, 5))
for i in range(10):
    plt.subplot(2, 5, i + 1); plt.imshow(x[i].numpy().astype("uint8")); plt.axis("off")
    ok = p[i].argmax() == y[i]
    plt.title(f"{klases[p[i].argmax()]} {p[i].max():.0%}", color="green" if ok else "red")
plt.suptitle("Prognozės: žalia – teisinga, raudona – klaida"); plt.tight_layout(); plt.show()

## 5. Išvados

- CNN nuo nulio su ~2900 mokymo nuotraukų pasiekia apie **65–75 %** validacijos tikslumą.
- Augmentacija ir `Dropout` sumažina persimokymą: mokymo ir validacijos kreivės nenutolsta viena nuo kitos.
- Dažniausiai painiojamos panašios klasės (pvz. rožė ir tulpė).
- Norint daug didesnio tikslumo su tiek pat duomenų, naudojamas **transfer learning** (užduotis 1.5).